---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！

# 02. PyTorch 设备检查

本节将详细介绍如何检查和管理 PyTorch 的计算设备（CPU、CUDA GPU、Apple Silicon MPS）。

## 学习目标
- 检测可用的计算设备
- 了解不同设备的特点和性能
- 学习多GPU环境的管理
- 掌握设备选择的最佳实践

## 1. 导入必要的库

In [ ]:
import torch
import platform
import sys

print(f"Python 版本: {sys.version}")
print(f"PyTorch 版本: {torch.__version__}")
print(f"操作系统: {platform.system()} {platform.release()}")

## 2. CPU 设备检查

CPU 是最基础的计算设备，所有系统都支持。虽然速度较慢，但兼容性最好。

In [ ]:
# CPU 总是可用的
print("=" * 50)
print("CPU 信息")
print("=" * 50)

# CPU 核心数
print(f"逻辑 CPU 核心数: {torch.get_num_threads()}")

# CPU 架构信息
print(f"CPU 架构: {platform.machine()}")
print(f"处理器: {platform.processor()}")

## 3. CUDA GPU 检查

CUDA 是 NVIDIA GPU 的并行计算平台，适用于深度学习训练。

In [ ]:
print("=" * 50)
print("CUDA GPU 信息")
print("=" * 50)

# 检查 CUDA 是否可用
cuda_available = torch.cuda.is_available()
print(f"CUDA 是否可用: {cuda_available}")

if cuda_available:
    # CUDA 版本
    print(f"CUDA 版本: {torch.version.cuda}")
    
    # GPU 数量
    gpu_count = torch.cuda.device_count()
    print(f"GPU 数量: {gpu_count}")
    
    # 遍历所有 GPU
    for i in range(gpu_count):
        print(f"\n--- GPU {i} ---")
        print(f"名称: {torch.cuda.get_device_name(i)}")
        print(f"计算能力: {torch.cuda.get_device_capability(i)}")
        
        # 显存信息（单位：GB）
        total_memory = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"总显存: {total_memory:.2f} GB")
        
        # 当前显存使用情况
        allocated = torch.cuda.memory_allocated(i) / (1024**3)
        reserved = torch.cuda.memory_reserved(i) / (1024**3)
        print(f"已分配显存: {allocated:.2f} GB")
        print(f"已保留显存: {reserved:.2f} GB")
        print(f"可用显存: {total_memory - reserved:.2f} GB")
    
    # 当前默认 GPU
    print(f"\n当前默认 GPU: cuda:{torch.cuda.current_device()}")
else:
    print("未检测到 CUDA GPU 或 CUDA 未正确安装")
    print("提示: 如需使用 GPU 加速，请安装支持 CUDA 的 PyTorch 版本")

## 4. Apple Silicon MPS 检查

MPS (Metal Performance Shaders) 是 Apple Silicon (M1/M2/M3) 的 GPU 加速框架。

In [ ]:
print("=" * 50)
print("Apple MPS 信息")
print("=" * 50)

# 检查 MPS 是否可用
mps_available = torch.backends.mps.is_available()
print(f"MPS 是否可用: {mps_available}")

if mps_available:
    # MPS 是否已构建
    mps_built = torch.backends.mps.is_built()
    print(f"MPS 是否已构建: {mps_built}")
    
    # 检测 Apple Silicon 芯片
    if platform.machine() == 'arm64':
        print(f"检测到 Apple Silicon 芯片 (ARM64 架构)")
        print(f"建议使用 MPS 设备进行 GPU 加速训练")
else:
    print("未检测到 MPS 支持")
    if platform.system() == 'Darwin':
        print("提示: 如果您使用的是 Apple Silicon Mac，请确保：")
        print("  1. macOS 版本 >= 12.3")
        print("  2. PyTorch 版本 >= 1.12")
    else:
        print("MPS 仅在 Apple Silicon Mac 上可用")

## 5. 设备性能对比

通过简单的矩阵运算对比不同设备的性能。

In [ ]:
import time

def benchmark_device(device, size=5000, iterations=10):
    """
    在指定设备上进行性能测试
    
    Args:
        device: 测试设备
        size: 矩阵大小
        iterations: 迭代次数
    
    Returns:
        平均执行时间（秒）
    """
    try:
        # 创建随机矩阵
        a = torch.randn(size, size, device=device)
        b = torch.randn(size, size, device=device)
        
        # 预热（让设备准备好）
        for _ in range(3):
            _ = torch.matmul(a, b)
        
        # 同步（确保计算完成）
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elif device.type == 'mps':
            torch.mps.synchronize()
        
        # 开始计时
        start_time = time.time()
        
        # 执行矩阵乘法
        for _ in range(iterations):
            c = torch.matmul(a, b)
        
        # 同步
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elif device.type == 'mps':
            torch.mps.synchronize()
        
        # 结束计时
        end_time = time.time()
        
        avg_time = (end_time - start_time) / iterations
        return avg_time
    
    except Exception as e:
        print(f"设备 {device} 测试失败: {e}")
        return None

# 测试所有可用设备
print("=" * 50)
print("设备性能对比 (矩阵乘法 5000x5000)")
print("=" * 50)

results = {}

# CPU 测试
print("\n测试 CPU...")
cpu_time = benchmark_device(torch.device('cpu'))
if cpu_time:
    results['CPU'] = cpu_time
    print(f"CPU 平均时间: {cpu_time:.4f} 秒")

# CUDA 测试
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        device = torch.device(f'cuda:{i}')
        print(f"\n测试 {device} ({torch.cuda.get_device_name(i)})...")
        cuda_time = benchmark_device(device)
        if cuda_time:
            results[f'CUDA:{i}'] = cuda_time
            print(f"CUDA:{i} 平均时间: {cuda_time:.4f} 秒")
            if cpu_time:
                speedup = cpu_time / cuda_time
                print(f"相比 CPU 加速: {speedup:.2f}x")

# MPS 测试
if torch.backends.mps.is_available():
    print("\n测试 MPS...")
    mps_time = benchmark_device(torch.device('mps'))
    if mps_time:
        results['MPS'] = mps_time
        print(f"MPS 平均时间: {mps_time:.4f} 秒")
        if cpu_time:
            speedup = cpu_time / mps_time
            print(f"相比 CPU 加速: {speedup:.2f}x")

# 性能总结
if len(results) > 1:
    print("\n" + "=" * 50)
    print("性能排名（从快到慢）")
    print("=" * 50)
    sorted_results = sorted(results.items(), key=lambda x: x[1])
    for rank, (device_name, exec_time) in enumerate(sorted_results, 1):
        print(f"{rank}. {device_name}: {exec_time:.4f} 秒")

## 6. 多 GPU 管理

如果系统中有多个 GPU，可以灵活选择使用哪个 GPU。

In [ ]:
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    print("=" * 50)
    print("多 GPU 管理")
    print("=" * 50)
    
    # 列出所有 GPU
    print(f"\n检测到 {torch.cuda.device_count()} 个 GPU:")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    
    # 设置默认 GPU
    print(f"\n当前默认 GPU: cuda:{torch.cuda.current_device()}")
    
    # 切换 GPU
    if torch.cuda.device_count() > 1:
        torch.cuda.set_device(1)
        print(f"切换后默认 GPU: cuda:{torch.cuda.current_device()}")
        
        # 恢复到 GPU 0
        torch.cuda.set_device(0)
    
    # 在不同 GPU 上创建张量
    print("\n在不同 GPU 上创建张量:")
    tensor_gpu0 = torch.randn(3, 3, device='cuda:0')
    print(f"张量 1 所在设备: {tensor_gpu0.device}")
    
    tensor_gpu1 = torch.randn(3, 3, device='cuda:1')
    print(f"张量 2 所在设备: {tensor_gpu1.device}")
    
    # 跨 GPU 数据传输
    print("\n跨 GPU 数据传输:")
    tensor_transferred = tensor_gpu0.to('cuda:1')
    print(f"传输后张量所在设备: {tensor_transferred.device}")
    
else:
    print("系统中只有一个或没有 GPU，跳过多 GPU 管理演示")

## 7. 设备选择最佳实践

自动选择最优设备的通用函数。

In [ ]:
def get_best_device(prefer_cuda=True):
    """
    自动选择最优计算设备
    
    Args:
        prefer_cuda: 如果为 True，优先选择 CUDA；否则优先选择 MPS
    
    Returns:
        torch.device 对象
    """
    if prefer_cuda:
        # 优先级: CUDA > MPS > CPU
        if torch.cuda.is_available():
            device = torch.device('cuda')
            print(f"选择设备: {device} ({torch.cuda.get_device_name(0)})")
        elif torch.backends.mps.is_available():
            device = torch.device('mps')
            print(f"选择设备: {device} (Apple Silicon GPU)")
        else:
            device = torch.device('cpu')
            print(f"选择设备: {device}")
    else:
        # 优先级: MPS > CUDA > CPU
        if torch.backends.mps.is_available():
            device = torch.device('mps')
            print(f"选择设备: {device} (Apple Silicon GPU)")
        elif torch.cuda.is_available():
            device = torch.device('cuda')
            print(f"选择设备: {device} ({torch.cuda.get_device_name(0)})")
        else:
            device = torch.device('cpu')
            print(f"选择设备: {device}")
    
    return device

# 测试设备选择
print("=" * 50)
print("最佳设备选择")
print("=" * 50)

print("\n优先 CUDA:")
device_cuda_first = get_best_device(prefer_cuda=True)

print("\n优先 MPS:")
device_mps_first = get_best_device(prefer_cuda=False)

# 使用建议
print("\n" + "=" * 50)
print("使用建议")
print("=" * 50)
print("""
1. 训练大型模型时，优先使用 CUDA GPU（如果可用）
2. 在 Apple Silicon Mac 上，MPS 是很好的选择
3. 小型模型或测试代码可以使用 CPU
4. 始终在代码中显式指定设备，便于迁移和调试
5. 使用 .to(device) 方法将模型和数据移动到同一设备
""")

## 8. 设备检查总结

生成一个完整的设备检查报告。

In [ ]:
def generate_device_report():
    """
    生成完整的设备检查报告
    """
    print("=" * 60)
    print("PyTorch 设备检查报告")
    print("=" * 60)
    
    # 系统信息
    print("\n【系统信息】")
    print(f"  操作系统: {platform.system()} {platform.release()}")
    print(f"  Python 版本: {sys.version.split()[0]}")
    print(f"  PyTorch 版本: {torch.__version__}")
    
    # CPU 信息
    print("\n【CPU】")
    print(f"  ✓ 可用")
    print(f"  架构: {platform.machine()}")
    print(f"  线程数: {torch.get_num_threads()}")
    
    # CUDA 信息
    print("\n【CUDA】")
    if torch.cuda.is_available():
        print(f"  ✓ 可用")
        print(f"  CUDA 版本: {torch.version.cuda}")
        print(f"  cuDNN 版本: {torch.backends.cudnn.version()}")
        print(f"  GPU 数量: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"    GPU {i}: {torch.cuda.get_device_name(i)}")
            memory = torch.cuda.get_device_properties(i).total_memory / (1024**3)
            print(f"            显存: {memory:.2f} GB")
    else:
        print(f"  ✗ 不可用")
    
    # MPS 信息
    print("\n【MPS】")
    if torch.backends.mps.is_available():
        print(f"  ✓ 可用")
        print(f"  适用于 Apple Silicon GPU 加速")
    else:
        print(f"  ✗ 不可用")
    
    # 推荐设备
    print("\n【推荐设备】")
    best_device = get_best_device(prefer_cuda=True)
    print(f"  {best_device}")
    
    print("\n" + "=" * 60)

# 生成报告
generate_device_report()

## 练习题

1. 修改 `benchmark_device` 函数，测试不同矩阵大小的性能
2. 编写代码比较不同设备上的内存使用情况
3. 如果有多个 GPU，尝试在不同 GPU 上同时运行计算任务
4. 实现一个函数，根据可用显存自动选择最空闲的 GPU

## 下一步

在下一个 notebook 中，我们将学习如何创建第一个 PyTorch 张量，并在不同设备上进行操作。

---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！